# 02 — Data Cleaning

Proyecto **Coffee Rewards Offers** (Maven Analytics).

Aplicamos las 6 transformaciones documentadas en `openspec/specs/data-understanding/spec.md`
y generamos copias limpias en `data/processed/`. No se modifica `data/raw/`.

Decisiones aprobadas (change 0002):
- Aplanar `events.value` en columnas `offer_id` / `amount` / `reward`.
- Normalizar la clave `'offer id'` (con espacio) a `offer_id`.
- `offers.channels` se parsea y se guarda como JSON (opción A).
- Agregar flag derivado `has_demographics` a `customers_clean`.


## Setup

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd

RAW = Path("data/raw")
PROC = Path("data/processed")
PROC.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
print(f"pandas {pd.__version__}")

pandas 3.0.5


## 1) `customers` — demografía

In [2]:
customers = pd.read_csv(RAW / "customers.csv")
print("shape inicial:", customers.shape)

shape inicial: (17000, 5)


### 1a. `age=118` → NaN (centinela de faltante)

In [3]:
customers["age"] = customers["age"].replace(118, pd.NA).astype("Int64")
print("filas con age==118 antes:", (customers["age"].isna()).sum(), "(esperado 2175)")

filas con age==118 antes: 2175 (esperado 2175)


### 1b. `gender` / `income` vacíos → NaN (quedan explícitos)

In [4]:
customers["gender"] = customers["gender"].replace("", pd.NA)
customers["income"] = customers["income"].replace("", pd.NA)
print("gender nulos:", customers["gender"].isna().sum())
print("income nulos:", customers["income"].isna().sum())

gender nulos: 2175
income nulos: 2175


### 1c. `became_member_on` (yyyymmdd) → datetime

In [5]:
customers["became_member_on"] = pd.to_datetime(
    customers["became_member_on"].astype(str), format="%Y%m%d"
)
print(customers["became_member_on"].dtype)
print("min:", customers["became_member_on"].min(), " max:", customers["became_member_on"].max())

datetime64[us]
min: 2013-07-29 00:00:00  max: 2018-07-26 00:00:00


### 1d. Flag derivado `has_demographics`

Marca la cohorte que tiene datos demográficos completos (edad, género e ingreso).
La cohorte sin demografía (2.175 filas) coincide exactamente con el centinela `age=118`.


In [6]:
customers["has_demographics"] = (
    customers["age"].notna()
    & customers["gender"].notna()
    & customers["income"].notna()
)
print(customers["has_demographics"].value_counts().to_string())

has_demographics
True     14825
False     2175


In [7]:
customers.dtypes

customer_id                    str
became_member_on    datetime64[us]
gender                         str
age                          Int64
income                     float64
has_demographics              bool
dtype: object

### 1e. Guardar `customers_clean.csv`

In [8]:
customers.to_csv(PROC / "customers_clean.csv", index=False)
print("guardado: customers_clean.csv", customers.shape)

guardado: customers_clean.csv (17000, 6)


---

## 2) `offers` — catálogo de ofertas

In [9]:
offers = pd.read_csv(RAW / "offers.csv")
print("shape inicial:", offers.shape)

shape inicial: (10, 6)


### 2a. `channels` → parsear lista Python (opción A: JSON)

In [10]:
offers["channels"] = offers["channels"].map(ast.literal_eval)
print("tipo tras parsear:", type(offers["channels"].iloc[0]))
offers["channels_json"] = offers["channels"].map(json.dumps)
offers = offers.drop(columns=["channels"]).rename(columns={"channels_json": "channels"})
print(offers[["offer_id", "channels"]].to_string(index=False))

tipo tras parsear: <class 'list'>
                        offer_id                             channels
ae264e3637204a6fb9bb56bc8210ddfd        ["email", "mobile", "social"]
4d5c57ea9a6940dd891ad53e9dbe8da0 ["web", "email", "mobile", "social"]
3f207df678b143eea3cee63160fa8bed           ["web", "email", "mobile"]
9b98b8c7a33c4b65b9aebfe6a799e6d9           ["web", "email", "mobile"]
0b1e1539f2cc45b7b9fa7c272da2e1d7                     ["web", "email"]
2298d6c36e964ae4a3e7e9706d1fb8c2 ["web", "email", "mobile", "social"]
fafdcd668e3743c1bb461111dcafc2a4 ["web", "email", "mobile", "social"]
5a8bc65990b245e5a138643cd4eb9837        ["email", "mobile", "social"]
f19421c1d4aa40978ebb69ca19b0e20d ["web", "email", "mobile", "social"]
2906b810c7d4411798c6938adc9daaa5           ["web", "email", "mobile"]


### 2b. Guardar `offers_clean.csv`

In [11]:
offers.to_csv(PROC / "offers_clean.csv", index=False)
print("guardado: offers_clean.csv", offers.shape)

guardado: offers_clean.csv (10, 6)


---

## 3) `events` — actividad de clientes

In [12]:
events = pd.read_csv(RAW / "events.csv")
print("shape inicial:", events.shape)

shape inicial: (306534, 4)


### 3a. `value` → parsear dict y aplanar en `offer_id` / `amount` / `reward`

Las claves son inconsistentes: `'offer id'` (espacio) en received/viewed y
`'offer_id'` (guion bajo) en completed. Normalizamos ambas a `offer_id`.


In [13]:
vals = events["value"].map(ast.literal_eval)  # Series de dicts

offer_id = vals.map(lambda d: d.get("offer id") if "offer id" in d else d.get("offer_id"))
amount = vals.map(lambda d: d.get("amount"))
reward = vals.map(lambda d: d.get("reward"))

events_clean = pd.DataFrame({
    "customer_id": events["customer_id"],
    "event": events["event"],
    "offer_id": offer_id,
    "amount": amount,
    "reward": reward,
    "time": events["time"],
})
print("shape limpio:", events_clean.shape)
print("columnas:", list(events_clean.columns))

shape limpio: (306534, 6)
columnas: ['customer_id', 'event', 'offer_id', 'amount', 'reward', 'time']


In [14]:
events_clean.head(8)

,customer_id,event,offer_id,amount,reward,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,0
5,389bc3fa690240e798340f5a15918d5c,offer received,f19421c1d4aa40978ebb69ca19b0e20d,NaN,NaN,0
6,c4863c7985cf408faee930f111475da3,offer received,2298d6c36e964ae4a3e7e9706d1fb8c2,NaN,NaN,0
7,2eeac8d8feae4a8cad5a6af0499a211d,offer received,3f207df678b143eea3cee63160fa8bed,NaN,NaN,0


### 3b. Guardar `events_clean.csv`

In [15]:
events_clean.to_csv(PROC / "events_clean.csv", index=False)
print("guardado: events_clean.csv", events_clean.shape)

guardado: events_clean.csv (306534, 6)


---

## 4) Validación

### 4a. Conteos de filas preservados

In [16]:
print("customers:", customers.shape[0], "(esperado 17000)")
print("offers:   ", offers.shape[0], "(esperado 10)")
print("events:   ", events_clean.shape[0], "(esperado 306534)")

customers: 17000 (esperado 17000)
offers:    10 (esperado 10)
events:    306534 (esperado 306534)


### 4b. Nulos esperados

In [17]:
print("customers nulos:")
print(customers.isna().sum().to_string())
print()
print("events nulos (esperado: offer_id=138953, amount=167581, reward=272955):")
print(events_clean.isna().sum().to_string())

customers nulos:
customer_id            0
became_member_on       0
gender              2175
age                 2175
income              2175
has_demographics       0

events nulos (esperado: offer_id=138953, amount=167581, reward=272955):
customer_id         0
event               0
offer_id       138953
amount         167581
reward         272955
time                0


### 4c. Distribución de eventos conservada

In [18]:
print(events_clean["event"].value_counts().to_string())

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579


### 4d. Integridad referencial: `offer_id` de events ⊆ catálogo de offers

In [19]:
catalog = set(offers["offer_id"])
in_catalog = events_clean["offer_id"].dropna().isin(catalog)
print("offer_id en events:", events_clean["offer_id"].notna().sum())
print("offer_id fuera del catálogo:", (~in_catalog).sum(), "(esperado 0)")

offer_id en events: 167581
offer_id fuera del catálogo: 0 (esperado 0)


### 4e. Rangos de `amount` y `reward`

In [20]:
print("amount: min =", events_clean["amount"].min(), " max =", events_clean["amount"].max())
print("reward unique:", sorted(events_clean["reward"].dropna().unique()))
print("time: min =", events_clean["time"].min(), " max =", events_clean["time"].max())

amount: min = 0.05  max = 1062.28
reward unique: [np.float64(2.0), np.float64(3.0), np.float64(5.0), np.float64(10.0)]
time: min = 0  max = 714


### 4f. Sanity: la cohorte sin demografía no rompe los joins

Los 2.175 clientes sin demografía siguen presentes en `customers_clean` (con
`has_demographics=False`) y sus eventos se conservan en `events_clean`.


In [21]:
customers_ids = set(customers["customer_id"])
events_ids = set(events_clean["customer_id"])
print("clientes en events que no están en customers:", len(events_ids - customers_ids), "(esperado 0)")

clientes en events que no están en customers: 0 (esperado 0)


---

## Resumen de la etapa

- `customers_clean.csv` — 17.000 × 6 (`age` centinela→NaN, `became_member_on`→datetime, flag `has_demographics`).
- `offers_clean.csv` — 10 × 6 (`channels` parseado a JSON).
- `events_clean.csv` — 306.534 × 6 (`value` aplanado en `offer_id`/`amount`/`reward`, clave normalizada).
- Validaciones OK: conteos, nulos, rangos e integridad referencial.
